# Comparação de abordagens fuzzy no pipeline coronário

Este notebook compara baselines e variações focadas em três perguntas práticas:

- o threshold fuzzy alpha-cut melhora quando fica menos expansivo?
- o contextual fuzzy melhora quando penaliza densos com mais força no vesselness arterial?
- a fuzzy connectedness melhora quando fica mais permissiva ou mais estrita que a configuração base?

Aqui, `contextual fuzzy` é a abordagem que calcula pertinências locais e usa o mapa fuzzy para ponderar o vesselness. A abordagem `fuzzy threshold` é separada: ela usa alpha-cut para decidir quais voxels entram como objeto no threshold inicial.


## 1. Ambiente

Carrega os helpers modularizados da comparação fuzzy. As células abaixo não executam nada pesado até a seção de execução.


In [ ]:
# ruff: noqa: E402
import sys
import time
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config" / "pipeline_config.json").exists():
            return candidate
    raise RuntimeError("Não encontrei a raiz do repositório a partir do cwd atual.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments.fuzzy_pipeline_comparison import (
    build_base_config,
    parameter_row,
    run_image,
    save_outputs,
    split_overrides,
    summarize_variant,
)
from utils.experiments.sweep_common import apply_overrides, sanitize_name, select_ids
from utils.project.config import scale_config_to_resolution
from utils.project.notebook_env import resolve_imagecas_base_path

## 2. Configuração da rodada

Ajuste aqui o split, quantidade de imagens e GPU/cache. Para comparar rapidamente, comece com poucas imagens.

In [ ]:
SPLIT = "train"          # train, val ou test
SAMPLE_SIZE = 30          # aumente quando os testes pequenos estiverem estáveis
START_INDEX = 0
IDS = None               # exemplo: "13,48,965"; se None, usa o split fixo
RESOLUTION = "mid"       # mid ou high
USE_GPU = None           # None respeita o config; True/False força
LOAD_CACHE = False
SAVE_CACHE = False

RUN_NAME = "fuzzy_pipeline_tuning"
CONFIG_PATH = REPO_ROOT / "config" / "pipeline_config.json"
BASE_PATH = resolve_imagecas_base_path()
RUN_DIR = REPO_ROOT / "output" / "segmentation" / "analysis" / "fuzzy_pipeline_comparison" / RUN_NAME
CACHE_DIR = RUN_DIR / "cache"
RUN_DIR.mkdir(parents=True, exist_ok=True)

args = SimpleNamespace(
    config_path=CONFIG_PATH,
    resolution=RESOLUTION,
    load_cache=LOAD_CACHE,
    save_cache=SAVE_CACHE,
    use_gpu=USE_GPU,
)
base_config = build_base_config(args)
image_ids = select_ids(SPLIT, SAMPLE_SIZE, START_INDEX, IDS, BASE_PATH)

settings_df = pd.DataFrame(
    [
        {
            "split": SPLIT,
            "sample_size": SAMPLE_SIZE,
            "start_index": START_INDEX,
            "ids": image_ids,
            "resolution": RESOLUTION,
            "use_gpu": base_config.get("USE_GPU"),
            "load_cache": LOAD_CACHE,
            "save_cache": SAVE_CACHE,
            "run_dir": str(RUN_DIR),
        }
    ]
)
display(settings_df)

## 3. Abordagens

Esta seção mantém os baselines fortes e adiciona variações pequenas para entender três pontos: threshold fuzzy mais conservador, contextual fuzzy realmente atuando no vesselness e FC mais permissivo/estrito.

`contextual_apply_to='artery'` pondera só o vesselness arterial. `contextual_apply_to='both'` também pondera o vesselness usado na escolha dos óstios, então pode mudar a localização dos óstios.


In [ ]:
FC_PARAMS = {
    "fc.alpha": 0.18,
    "fc.sigma_hu": 80,
    "fc.neighborhood": 26,
    "fc.candidate_min_vesselness": 0.02,
    "fc.seed_search_radius": 2,
    "fc.max_seeds_per_ostium": 4,
    "fc.seed_min_vesselness": 0.02,
    "fc.min_seed_distance_voxels": 1.0,
    "fc.vesselness_affinity_mode": "geometric_mean",
    "fc.vesselness_floor": 0.02,
    "fc.edge_affinity_mode": "weighted_product",
    "fc.vesselness_weight": 0.9,
    "fc.mask_strategy": "alpha",
}

FC_PERMISSIVE_PARAMS = {
    **FC_PARAMS,
    # Tenta recuperar casos em que FC subsegmenta ou para cedo demais.
    "fc.alpha": 0.14,
    "fc.sigma_hu": 120,
    "fc.candidate_min_vesselness": 0.015,
    "fc.seed_min_vesselness": 0.015,
    "fc.vesselness_floor": 0.015,
}

FC_STRICT_PARAMS = {
    **FC_PARAMS,
    # Tenta reduzir vazamentos em casos em que a segmentação cresce demais.
    "fc.alpha": 0.22,
    "fc.sigma_hu": 60,
    "fc.candidate_min_vesselness": 0.03,
    "fc.seed_min_vesselness": 0.02,
    "fc.vesselness_floor": 0.03,
}

CONTEXTUAL_PARAMS = {
    "contextual.weight_floor": 0.15,
    "contextual.dense_power": 2.0,
    "contextual.weight_mode": "dense_only",
    "contextual.soft_margin_hu": 160,
    "contextual.object_percentile": 99.8,
    "contextual.dense_percentile": 99.95,
    "contextual.smooth_radius": 1,
    "contextual.smooth_mode": "mean",
}

CONTEXTUAL_STRONG_PARAMS = {
    # Mais agressivo que o contextual original, que ficou quase neutro.
    "contextual.weight_floor": 0.05,
    "contextual.dense_power": 4.0,
    "contextual.weight_mode": "object_dense",
    "contextual.soft_margin_hu": 160,
    "contextual.object_percentile": 99.5,
    "contextual.dense_percentile": 99.9,
    "contextual.smooth_radius": 2,
    "contextual.smooth_mode": "mean",
}

FUZZY_THRESHOLD_PARAMS = {
    "threshold_mode": "fuzzy_alpha",
    "threshold_alpha": 0.5,
    "threshold_margin_hu": 160,
}

FUZZY_THRESHOLD_CONSERVATIVE_PARAMS = {
    # Menos expansão da máscara que o fuzzy alpha-cut original.
    # Ajuda a testar casos em que fuzzy_threshold mudou a localização dos óstios.
    "threshold_mode": "fuzzy_alpha",
    "threshold_alpha": 0.7,
    "threshold_margin_hu": 80,
}

FUZZY_THRESHOLD_BALANCED_PARAMS = {
    # Intermediário entre o normal e o fuzzy atual.
    "threshold_mode": "fuzzy_alpha",
    "threshold_alpha": 0.6,
    "threshold_margin_hu": 120,
}

VARIANTS = [
    # Baselines fortes da rodada anterior.
    {
        "name": "normal_rg",
        "description": "Threshold HU normal + region growing.",
        "threshold_rule": "-300 <= I <= P99.7",
        "vesselness_rule": "sem ponderação contextual",
        "overrides": {
            "threshold_mode": "normal",
            "contextual_apply_to": "none",
            "artery_method": "region_growing",
        },
    },
    {
        "name": "fuzzy_threshold_rg",
        "description": "Fuzzy alpha-cut original no threshold + region growing.",
        "threshold_rule": "pertinência fuzzy >= 0.5, margem 160 HU",
        "vesselness_rule": "sem ponderação contextual",
        "overrides": {
            **FUZZY_THRESHOLD_PARAMS,
            "contextual_apply_to": "none",
            "artery_method": "region_growing",
        },
    },
    {
        "name": "normal_threshold_fc",
        "description": "Threshold HU normal + fuzzy connectedness arterial.",
        "threshold_rule": "-300 <= I <= P99.7",
        "vesselness_rule": "sem ponderação contextual",
        "overrides": {
            "threshold_mode": "normal",
            "contextual_apply_to": "none",
            "artery_method": "fuzzy_connectedness",
            **FC_PARAMS,
        },
    },
    {
        "name": "fuzzy_threshold_fc",
        "description": "Fuzzy alpha-cut original no threshold + fuzzy connectedness arterial.",
        "threshold_rule": "pertinência fuzzy >= 0.5, margem 160 HU",
        "vesselness_rule": "sem ponderação contextual",
        "overrides": {
            **FUZZY_THRESHOLD_PARAMS,
            "contextual_apply_to": "none",
            "artery_method": "fuzzy_connectedness",
            **FC_PARAMS,
        },
    },

    # Fuzzy threshold: testa versões menos expansivas para não deslocar óstios/aorta.
    {
        "name": "fuzzy_threshold_balanced_rg",
        "description": "Fuzzy threshold intermediário + region growing.",
        "threshold_rule": "pertinência fuzzy >= 0.6, margem 120 HU",
        "vesselness_rule": "sem ponderação contextual",
        "overrides": {
            **FUZZY_THRESHOLD_BALANCED_PARAMS,
            "contextual_apply_to": "none",
            "artery_method": "region_growing",
        },
    },
    {
        "name": "fuzzy_threshold_conservative_rg",
        "description": "Fuzzy threshold conservador + region growing.",
        "threshold_rule": "pertinência fuzzy >= 0.7, margem 80 HU",
        "vesselness_rule": "sem ponderação contextual",
        "overrides": {
            **FUZZY_THRESHOLD_CONSERVATIVE_PARAMS,
            "contextual_apply_to": "none",
            "artery_method": "region_growing",
        },
    },
    {
        "name": "fuzzy_threshold_balanced_fc",
        "description": "Fuzzy threshold intermediário + fuzzy connectedness.",
        "threshold_rule": "pertinência fuzzy >= 0.6, margem 120 HU",
        "vesselness_rule": "sem ponderação contextual",
        "overrides": {
            **FUZZY_THRESHOLD_BALANCED_PARAMS,
            "contextual_apply_to": "none",
            "artery_method": "fuzzy_connectedness",
            **FC_PARAMS,
        },
    },

    # Contextual fuzzy: versões fortes para deixar de ser quase neutro.
    {
        "name": "contextual_strong_rg",
        "description": "Threshold normal + contextual fuzzy mais agressivo no vesselness arterial + RG.",
        "threshold_rule": "-300 <= I <= P99.7",
        "vesselness_rule": "penaliza densos e reforça objeto no vesselness_artery",
        "overrides": {
            "threshold_mode": "normal",
            "contextual_apply_to": "artery",
            "artery_method": "region_growing",
            **CONTEXTUAL_STRONG_PARAMS,
        },
    },
    {
        "name": "contextual_strong_fc",
        "description": "Threshold normal + contextual fuzzy mais agressivo no vesselness arterial + FC.",
        "threshold_rule": "-300 <= I <= P99.7",
        "vesselness_rule": "penaliza densos e reforça objeto no vesselness_artery",
        "overrides": {
            "threshold_mode": "normal",
            "contextual_apply_to": "artery",
            "artery_method": "fuzzy_connectedness",
            **CONTEXTUAL_STRONG_PARAMS,
            **FC_PARAMS,
        },
    },
    {
        "name": "contextual_strong_both_rg",
        "description": "Contextual fuzzy também no vesselness dos óstios + RG.",
        "threshold_rule": "-300 <= I <= P99.7",
        "vesselness_rule": "pondera vesselness_ostios e vesselness_artery",
        "overrides": {
            "threshold_mode": "normal",
            "contextual_apply_to": "both",
            "artery_method": "region_growing",
            **CONTEXTUAL_STRONG_PARAMS,
        },
    },

    # FC: versões para entender quando FC deve ser mais permissivo ou mais estrito.
    {
        "name": "normal_fc_permissive",
        "description": "Threshold normal + FC mais permissivo.",
        "threshold_rule": "-300 <= I <= P99.7",
        "vesselness_rule": "sem ponderação contextual",
        "overrides": {
            "threshold_mode": "normal",
            "contextual_apply_to": "none",
            "artery_method": "fuzzy_connectedness",
            **FC_PERMISSIVE_PARAMS,
        },
    },
    {
        "name": "normal_fc_strict",
        "description": "Threshold normal + FC mais estrito.",
        "threshold_rule": "-300 <= I <= P99.7",
        "vesselness_rule": "sem ponderação contextual",
        "overrides": {
            "threshold_mode": "normal",
            "contextual_apply_to": "none",
            "artery_method": "fuzzy_connectedness",
            **FC_STRICT_PARAMS,
        },
    },
]

variant_overview_df = pd.DataFrame(
    [
        {
            "variant": item["name"],
            "description": item["description"],
            "threshold_rule": item["threshold_rule"],
            "vesselness_rule": item["vesselness_rule"],
            "overrides": item["overrides"],
        }
        for item in VARIANTS
    ]
)
display(variant_overview_df)


## 4. Execução

Esta célula roda o pipeline para cada imagem e abordagem. Resultados parciais são salvos em CSV a cada abordagem.

In [ ]:
summaries = []
image_rows = []
parameter_rows = []

for variant_index, variant in enumerate(VARIANTS, start=1):
    variant_name = sanitize_name(variant["name"])
    overrides = variant.get("overrides", {})
    config_overrides, experiment = split_overrides(overrides)
    config = apply_overrides(base_config, config_overrides)
    config = scale_config_to_resolution(config)
    parameter_rows.append(parameter_row(variant_name, overrides, config, experiment))

    print(f"[{variant_index}/{len(VARIANTS)}] {variant_name}")
    start_time = time.time()
    variant_rows = []
    for img_index, img_id in enumerate(image_ids, start=1):
        print(f"  [{img_index}/{len(image_ids)}] IMG_ID={img_id}")
        row = run_image(
            img_id,
            variant_name,
            SPLIT,
            BASE_PATH,
            CACHE_DIR,
            config,
            experiment,
        )
        variant_rows.append(row)
        image_rows.append(row)

    runtime_seconds = time.time() - start_time
    summaries.append(summarize_variant(variant_name, variant_rows, runtime_seconds))
    save_outputs(RUN_DIR, summaries, image_rows, parameter_rows)

summary_df = pd.DataFrame(summaries).sort_values(
    ["selection_score", "ostia_success_rate", "mean_dice_success_ostia", "mean_dice"],
    ascending=[False, False, False, False],
    na_position="last",
)
image_results_df = pd.DataFrame(image_rows)
parameter_df = pd.DataFrame(parameter_rows)

display(summary_df)
print(f"CSV ranking: {RUN_DIR / 'summary' / 'ranking.csv'}")
print(f"CSV por imagem: {RUN_DIR / 'results' / 'image_results.csv'}")
print(f"CSV parâmetros: {RUN_DIR / 'parameters' / 'variant_parameters.csv'}")

## 5. Resumo por abordagem

Ranking ordenado por `selection_score`, que combina sucesso dos óstios com Dice nos casos em que os óstios foram corretos/toleráveis.

In [ ]:
summary_path = RUN_DIR / "summary" / "ranking.csv"
summary_df = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame(summaries)
display(summary_df)

## 6. Dice por imagem

Tabela larga para ver em quais imagens cada abordagem melhora ou piora.

In [ ]:
results_path = RUN_DIR / "results" / "image_results.csv"
image_results_df = pd.read_csv(results_path) if results_path.exists() else pd.DataFrame(image_rows)

dice_by_image_df = (
    image_results_df.pivot_table(
        index=["split", "IMG_ID", "ostia_status", "ostia_success"],
        columns="variant",
        values="dice_artery",
        aggfunc="first",
    )
    .reset_index()
)
display(dice_by_image_df)

## 6.1. Onde FC e RG divergem

Estas tabelas ajudam a separar os casos em que FC parece resolver vazamento/subsegmentação do RG e os casos em que RG ainda é melhor. Use isso para escolher se vale testar parâmetros mais permissivos ou mais restritivos.


In [ ]:
comparison_pairs = [
    ("normal_rg", "normal_threshold_fc"),
    ("fuzzy_threshold_rg", "fuzzy_threshold_fc"),
]

for rg_variant, fc_variant in comparison_pairs:
    if {rg_variant, fc_variant}.issubset(set(image_results_df["variant"])):
        pair_df = image_results_df[image_results_df["variant"].isin([rg_variant, fc_variant])]
        wide = pair_df.pivot_table(
            index=["IMG_ID"],
            columns="variant",
            values="dice_artery",
            aggfunc="first",
        ).reset_index()
        wide["fc_minus_rg"] = wide[fc_variant] - wide[rg_variant]
        status_df = image_results_df[image_results_df["variant"] == rg_variant][["IMG_ID", "ostia_status", "left_dist_mm", "right_dist_mm"]]
        wide = wide.merge(status_df, on="IMG_ID", how="left")
        print(f"\n{fc_variant} - {rg_variant}: FC melhor")
        display(wide.sort_values("fc_minus_rg", ascending=False).head(8))
        print(f"\n{fc_variant} - {rg_variant}: RG melhor")
        display(wide.sort_values("fc_minus_rg", ascending=True).head(8))


## 7. Threshold, contextual fuzzy, FC e óstios

Use esta tabela para entender se a mudança veio do threshold inicial, da ponderação contextual do vesselness arterial ou da troca do método arterial para fuzzy connectedness.


In [ ]:
important_columns = [
    "variant",
    "IMG_ID",
    "threshold_mode",
    "contextual_apply_to",
    "artery_method",
    "threshold_voxels",
    "lcc_voxels",
    "mean_contextual_weight",
    "ostia_status",
    "ostia_success",
    "left_dist_mm",
    "right_dist_mm",
    "dice_artery",
    "artery_voxels",
    "fc_processed_voxels",
    "fc_effective_alpha",
    "error",
]
existing_columns = [column for column in important_columns if column in image_results_df.columns]
display(image_results_df[existing_columns].sort_values(["IMG_ID", "variant"]))


## 8. Parâmetros usados

Registro compacto das configurações efetivas por abordagem.

In [ ]:
parameter_path = RUN_DIR / "parameters" / "variant_parameters.csv"
parameter_df = pd.read_csv(parameter_path) if parameter_path.exists() else pd.DataFrame(parameter_rows)
display(parameter_df)